# La motxilla

El problema de la motxilla és un dels problemes més coneguts en l'àmbit de la computació. El problema consisteix en omplir una motxilla amb objectes de diferent pes i valor, de manera que es maximitzi el valor de la motxilla sense excedir el pes màxim que pot suportar.

Per exemple, suposem que tenim una motxilla amb una capacitat de 100 kg i els següents objectes:

| Objecte   | Pes (kg) | Valor (€) |
|-----------|----------|-----------|
| Coca-Cola | 1        | 1.5       |
| Cervesa   | 2        | 2.5       |
| Aigua     | 1        | 0.5       |
| Cacauets  | 1        | 1.0       |
| Patates   | 2        | 1.5       |
| Galetes   | 1        | 1.0       |
| Pomes     | 1        | 1.0       |
| Taronges  | 1        | 1.0       |
| Plàtans   | 1        | 1.0       |
| Peres     | 1        | 1.0       |
| Maduixes  | 1        | 1.0       |
| Pinya     | 1        | 1.0       |
| Pomes     | 1        | 1.0       |

Quina combinació d'objectes podem posar a la motxilla per maximitzar el valor sense excedir el pes màxim?

El catàleg té dues entrades de pomes amb el mateix pes i valor. Les conservem com a dos lots separats, cadascun amb fins a 10 unitats.

Un individu és una tupla amb una quantitat de 0 a 10 per cada fila de la taula. Per exemple, `(0, 2, 0, …)` indica que agafem dues cerveses. El pes i el valor totals se sumen multiplicant cada quantitat pel pes i el valor d’una unitat.

In [ ]:
from random import Random

PES_MAXIM = 100
MAX_UNITATS = 10
objectes = [
    ('Coca-Cola', 1, 1.5),
    ('Cervesa', 2, 2.5),
    ('Aigua', 1, 0.5),
    ('Cacauets', 1, 1.0),
    ('Patates', 2, 1.5),
    ('Galetes', 1, 1.0),
    ('Pomes', 1, 1.0),
    ('Taronges', 1, 1.0),
    ('Plàtans', 1, 1.0),
    ('Peres', 1, 1.0),
    ('Maduixes', 1, 1.0),
    ('Pinya', 1, 1.0),
    ('Pomes', 1, 1.0),
]

def pes(individu):
    return sum(q * p for q, (_, p, _) in zip(individu, objectes))

def valor(individu):
    return sum(q * v for q, (_, _, v) in zip(individu, objectes))

def puntuacio(individu):
    if pes(individu) > PES_MAXIM:
        return -1
    return valor(individu)

## D'una generació a la següent
Ordenem la població segons la puntuació. Conservem el millor individu i creem la resta de descendents creuant parelles de la meitat superior. Una mutació canvia la quantitat d’un producte. La motxilla buida és el punt de partida factible de la població.

In [ ]:
def creua(a, b, rng):
    tall = rng.randrange(1, len(a))
    return a[:tall] + b[tall:]

def muta(individu, rng):
    i = rng.randrange(len(individu))
    opcions = [q for q in range(MAX_UNITATS + 1) if q != individu[i]]
    quantitat = rng.choice(opcions)
    return individu[:i] + (quantitat,) + individu[i + 1:]

def genetic(rng, generacions=30):
    poblacio = [(0,) * len(objectes)]
    for _ in range(19):
        poblacio.append(tuple(rng.randrange(MAX_UNITATS + 1) for _ in objectes))

    for _ in range(generacions):
        poblacio.sort(key=puntuacio, reverse=True)
        pares = poblacio[:10]
        nova = [poblacio[0]]
        while len(nova) < len(poblacio):
            a, b = rng.sample(pares, 2)
            fill = creua(a, b, rng)
            if rng.random() < 0.25:
                fill = muta(fill, rng)
            nova.append(fill)
        poblacio = nova
    return max(poblacio, key=puntuacio)

## Resultat
Mostrem els productes triats, el pes i el valor. Canvia la llavor o el nombre de generacions i compara els resultats.

In [ ]:
resultat = genetic(Random(42))
for quantitat, (nom, _, _) in zip(resultat, objectes):
    if quantitat:
        print(quantitat, nom)
print(f"{pes(resultat)} kg · {valor(resultat):.2f} €")

## Comprovació opcional de l’òptim
Hi ha 11 quantitats possibles per cadascuna de les 13 entrades. Per calcular l’òptim sense enumerar totes les combinacions, guardem el millor valor per a cada capacitat de 0 a 100 kg. Recorrem les capacitats de major a menor cada vegada que afegim una unitat, de manera que només es puga comptar una vegada.

In [ ]:
millor_valor = [0] * (PES_MAXIM + 1)
for _, p, v in objectes:
    for _ in range(MAX_UNITATS):
        for capacitat in range(PES_MAXIM, p - 1, -1):
            millor_valor[capacitat] = max(
                millor_valor[capacitat], millor_valor[capacitat - p] + v
            )
print(f"{millor_valor[PES_MAXIM]:.2f} €")